##Creation de la Base de donnée bronze_db

In [0]:
%sql
-- Création de la base de donnée
CREATE DATABASE IF NOT EXISTS bronze_db;

-- Affiche de la base de donnée
SHOW DATABASES;

In [0]:
%sql
-- utilisation de la base de donnée bronze_db
USE bronze_db;
 -- selectionné la base de donnée utilisée
 SELECT current_database()

In [0]:
%sql
--upprimer la table raw_trips si elle existe en sql
--DROP TABLE IF EXISTS raw_trips;

In [0]:
%python
# voir le nombre de fichier dans le volumes nyc_taxi
display(
    dbutils.fs.ls("/Volumes/workspace/default/nyc_taxi/bronze_data/")
      )

### Script d'ingestion des données sources

- Définir explicitement le schema des données
- Script d'ingestion_timestamp avec les condions suivants:
- Ne crée pas de nouvelle table si la table raw_trips exist
- Ne supprime pas les anciennes données de la table raw_trips
- Ajoute une colonne ingestion_timetamp dans la table
- s'il n'ya pas de nouveaux fichier parquet, ne fait rien
- Pour ajouter de nouvelle donnée de fichier parquet, compare le timestamp de la derniere     
  modification dans le 
- fichier parquet et la date de la derniere modification dans la table raw_trips
- Si le timestamp de la derniere modification dans le fichier parquet est plus recent que la date de  
  la derniere 
- modification dans la table raw_trips, ajoute les nouvelles données dans la table raw_trips
- Sinon, pas d'action
- il faut que le type des variables doit correspondre au type de la colonne dans la table raw_trips
- Si la colonne n'existe pas, il faut la créer dans la table raw_trips
- Si la colonne existe, il faut la mettre à jour dans la table raw_trips 

In [0]:

%python
from pyspark.sql.functions import current_timestamp,col,substring_index
from pyspark.sql.types import StructType,TimestampType, StringType, DoubleType, IntegerType,LongType, StructField
import datetime

# 1. Définir explicitement le schema des données
schema = StructType([
        StructField("vendorID", IntegerType(), True),
        StructField("tpep_pickup_datetime", TimestampType(), True),
        StructField("tpep_dropoff_datetime", TimestampType(), True),
        StructField("passenger_count", LongType(), True),
        StructField("trip_distance", DoubleType(), True),
        StructField("RatecodeID",LongType(), True),
        StructField("store_and_fwd_flag", StringType(), True),
        StructField("PULocationID", IntegerType(), True),
        StructField("DOLocationID", IntegerType(), True),
        StructField("payment_type", LongType(), True),
        StructField("fare_amount", DoubleType(), True),
        StructField("extra", DoubleType(), True),
        StructField("mta_tax", DoubleType(), True),
        StructField("tip_amount", DoubleType(), True),
        StructField("tolls_amount", DoubleType(), True),
        StructField("improvement_surcharge", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
        StructField("congestion_surcharge", DoubleType(), True),
        StructField("Airport_fee", DoubleType(), True)
])

# 2. Dossier source
source_folder = "/Volumes/workspace/default/nyc_taxi/bronze_data/"
archive_folder = source_folder + "archive/run_" + datetime.datetime.now().strftime("%Y%m%d%H%M%S") +"/"

# 3. Fichier parquet à lire dans la source_folder
filePath = source_folder + "*.parquet"

# 4. Lister les fichiers parquets qui sont dans le dossier source_folder
files = [f for f in dbutils.fs.ls(source_folder) if f.name.endswith(".parquet")]

# 5. si lire la liste files si elle n'est pas vide avec le schema forcé

if files:
        df = spark.read.schema(schema).parquet(filePath)
        # 1. Affichage du dataframe df
        df.show()
        # 2. Ajouter une colonne d'ingestion timestamp plus une colonne nom de fichier
        df = df.withColumn("ingestion_timetamp", current_timestamp()) \
            .withColumn("file_name", substring_index(col("_metadata.file_path"),"/", -1))
       #---------------------------
       # 3. calcule le nombre de ligne du fichier
       #-----------------------------
       # 3.1 nombre de ligne dans la dataframe à ingérer
        new_rows_count = df.count()
        df.show()
        # 3.2 verifier si la table parquet existe et compter son nombre de ligne existante
        if "raw_trips" in [t.name for t in spark.catalog.listTables("bronze_db")]:
                existing_rows_count = spark.table("raw_trips").count()
        else:
                existing_rows_count = 0
        # 3.3 Affichage des nombres de lignes
        print(f"Nombre de lignes à ingérer: {new_rows_count}")
        print(f"Nombre de lignes existantes dans la tables: {existing_rows_count}")
               
        # 4. Sauvegarde des données dans la tables Delta
        df.write.format("delta").mode("append").option("mergeSchema", True).saveAsTable("raw_trips")
        # 5. Contrôle de cohérence des nombres  de lignes
        final_rows_count = spark.table("raw_trips").count()
        expected_rows = existing_rows_count + new_rows_count
        
        if final_rows_count != expected_rows:
                raise Exception(f"Le contrôle de cohérence à échoué:Le nombre de lignes dans la table final {final_rows_count} ne correspond pas aux nombres de lignes attendues {expected_rows}")       
        else:
                print(f"Le contrôle de cohérence réussi:Le nombre de lignes dans la table final {final_rows_count} est égal aux nombres de lignes attendues {expected_rows}")
        # 5. Archibage des fichiers parquets traités
        archive_folder = source_folder + "archive/run_" + datetime.datetime.now().strftime("%Y%m%d%H%M%S") +"/"

        # Déplacer les fichiers parquets traiter dans le dossier archive_folder
        # Affichage du nom des fichiers
        for f in files:
                # deplace vers le répertoire archive
                dbutils.fs.mv(f.path, archive_folder + f.name)
                # Afficher la liste des fichiers archivés
                print(f"{len(files)}les fichiers ingérés et archivés dans{archive_folder}")
        # Afficher le contenu du dossier archive
        display(dbutils.fs.ls(archive_folder))
       
        # Afficher le contenu du dossier source_folder
        display(
        dbutils.fs.ls(archive_folder) )

else:
        print("Aucun fichier parquet trouvé dans le dossier source, il n'ya rien à ingérer")  



In [0]:
%sql
-- Affichage de la table raw_trips
SELECT * FROM raw_trips LIMIT 10;


In [0]:
%sql
-- supprimer la table raw_trips
--DROP TABLE raw_trips;


In [0]:
%python
# affichage des données dans bronze_db
dbutils.fs.ls("/Volumes/workspace/default/nyc_taxi/bronze_data/archive/")



In [0]:
%sql
-- Renommer le nom du volume 
-- ALTER VOLUME default.bronze_files
-- RENAME TO nyc_taxi;

-----------------------
-- Path_tripdata_01 = /Volum es/workspace/default/nyc_taxi/bronze_data/yellow_tripdata_2024-01.parquet/

-- Path_zone_taxi = /Volumes/workspace/default/nyc_taxi/bronze_data/taxi_zone_lookup.csv/